# Day 09. Exercise 00
# Regularization

## 0. Imports

In [ ]:
import sys
import time

import joblib
import pandas as pd
from IPython.core.display import Markdown
from IPython.display import HTML, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from tqdm.notebook import tqdm

In [2]:
notebook_start_time = time.time()

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.

In [3]:
df = pd.read_csv("../data/dayofweek.csv")
df.head(3)

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [4]:
X = df.drop(columns=["dayofweek"])
y = df["dayofweek"]

2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.

In [6]:
model_logreg = LogisticRegression(fit_intercept=False, random_state=21)

2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model

The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [7]:
%%time

skf = StratifiedKFold(n_splits=10)

cv_results = cross_validate(
    model_logreg,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)

mean_acc = cv_results["test_score"].mean()
std_acc = cv_results["test_score"].std()

for train_score, valid_score in zip(
    cv_results["train_score"], cv_results["test_score"]
):
    print(f"{'train - ':<10}{train_score:.5f}   |   {'valid - ':<10}{valid_score:.5f}")
print(f"""\033[32mAverage accuracy of Base Logistic Regression on crossval is {mean_acc:.5f}
Std is {std_acc:.5f}\033[0m""")

train -   0.62902   |   valid -   0.59259
train -   0.64633   |   valid -   0.62963
train -   0.63479   |   valid -   0.56296
train -   0.65622   |   valid -   0.61481
train -   0.63397   |   valid -   0.57778
train -   0.64056   |   valid -   0.59259
train -   0.64138   |   valid -   0.65926
train -   0.65952   |   valid -   0.56296
train -   0.64333   |   valid -   0.59701
train -   0.63674   |   valid -   0.62687
Average accuracy of Base Logistic Regression on crossval is 0.60165
Std is 0.02943
CPU times: user 75.8 ms, sys: 258 ms, total: 334 ms
Wall time: 622 ms


In [8]:
# cv_results

### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [9]:
%%time

common_solvers = ["lbfgs", "newton-cg", "sag", "saga"]
configs = {"none": common_solvers, "l2": common_solvers, "l1": ["saga"]}
results = []

for penalty, solvers in tqdm(configs.items(), file=sys.stderr):
    for solver in tqdm(solvers, leave=False, file=sys.stderr):
        model = LogisticRegression(
            penalty=penalty,
            solver=solver,
            fit_intercept=False,
            random_state=21,
            max_iter=2000,
        )

        cv_res = cross_validate(
            model, X_train, y_train, cv=skf, scoring="accuracy", n_jobs=-1
        )

        results.append(
            {
                "Penalty": "None" if penalty is None else penalty,
                "Solver": solver,
                "Mean CV Accuracy": round(cv_res["test_score"].mean(), 5),
                "Std": round(cv_res["test_score"].std(), 5),
            }
        )
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Penalty     Solver  Mean CV Accuracy      Std
   none      lbfgs           0.62462  0.03379
   none  newton-cg           0.62462  0.03379
   none        sag           0.62462  0.03379
   none       saga           0.62462  0.03379
     l2      lbfgs           0.60165  0.02943
     l2  newton-cg           0.60165  0.02943
     l2        sag           0.60165  0.02943
     l2       saga           0.60165  0.02943
     l1       saga           0.59646  0.02848
CPU times: user 204 ms, sys: 84.3 ms, total: 288 ms
Wall time: 7.74 s


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [10]:
model_svc_default = SVC(kernel="linear", random_state=21, probability=True)

In [11]:
%%time

cv_res_svc = cross_validate(
    model_svc_default,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)

mean_acc_svc = cv_res_svc["test_score"].mean()
std_acc_svc = cv_res_svc["test_score"].std()

for train_score, valid_score in zip(
    cv_res_svc["train_score"], cv_res_svc["test_score"]
):
    print(f"{'train - ':<10}{train_score:.5f}   |   {'valid - ':<10}{valid_score:.5f}")
print(f"""\033[32mAverage accuracy of Base SVM Model on crossval is {mean_acc_svc:.5f}
Std is {std_acc_svc:.5f}\033[0m""")

train -   0.70486   |   valid -   0.65926
train -   0.69662   |   valid -   0.75556
train -   0.69415   |   valid -   0.62222
train -   0.70239   |   valid -   0.65185
train -   0.69085   |   valid -   0.65185
train -   0.68920   |   valid -   0.64444
train -   0.69250   |   valid -   0.72593
train -   0.70074   |   valid -   0.62222
train -   0.69605   |   valid -   0.61940
train -   0.71087   |   valid -   0.63433
Average accuracy of Base SVM Model on crossval is 0.65871
Std is 0.04359
CPU times: user 12.1 ms, sys: 9.64 ms, total: 21.7 ms
Wall time: 365 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [12]:
%%time

c = [0.01, 0.1, 1, 10, 50, 100, 500]
results_svc = []

for reg_parameter in c:
    model_svc_optimized = SVC(
        kernel="linear", random_state=21, probability=True, C=reg_parameter
    )

    cv_res_svc = cross_validate(
        model_svc_optimized,
        X_train,
        y_train,
        cv=skf,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1,
    )
    mean_acc = cv_res_svc["test_score"].mean()
    std_acc = cv_res_svc["test_score"].std()

    results_svc.append(
        {
            "C": reg_parameter,
            "Mean accuracy": round(mean_acc, 5),
            "Std": round(std_acc, 5),
        }
    )
df_res_svc = pd.DataFrame(results_svc)
print(df_res_svc.to_string(index=False))

      C  Mean accuracy      Std
   0.01        0.37534  0.01848
   0.10        0.56230  0.02177
   1.00        0.65871  0.04359
  10.00        0.72771  0.04417
  50.00        0.75441  0.03438
 100.00        0.75589  0.03550
 500.00        0.75441  0.03357
CPU times: user 159 ms, sys: 60.5 ms, total: 220 ms
Wall time: 14.8 s


*Модель достигает своего максимума при **C = 100.00** с точностью **0.75589**. При дальнейшем увеличении C до **500.00** точность перестает расти и даже немного откатывается назад (до **0.75441**). Это сигнал того, что модель начинает избыточно подстраиваться под обучающую выборку. Стандартное отклонение заметно снизилось по сравнению с базовой моделью (упало с **0.04359** до **0.035**). Это значит, что модель стала не просто точнее, но и стабильнее ведет себя на разных фолдах.*

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [13]:
model_dt = DecisionTreeClassifier(max_depth=10, random_state=21)

In [14]:
%%time

cv_res_dt = cross_validate(
    model_dt,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)

mean_acc_dt = cv_res_dt["test_score"].mean()
std_acc_dt = cv_res_dt["test_score"].std()

for train_score, valid_score in zip(cv_res_dt["train_score"], cv_res_dt["test_score"]):
    print(f"{'train - ':<10}{train_score:.5f}   |   {'valid - ':<10}{valid_score:.5f}")
print(f"""\033[32mAverage accuracy of Base Decision Tree Model on crossval is {mean_acc_dt:.5f},
Std is {std_acc_dt:.5f}\033[0m""")

train -   0.81039   |   valid -   0.74074
train -   0.77741   |   valid -   0.74074
train -   0.83347   |   valid -   0.70370
train -   0.79720   |   valid -   0.76296
train -   0.82440   |   valid -   0.75556
train -   0.80379   |   valid -   0.68889
train -   0.80709   |   valid -   0.76296
train -   0.80132   |   valid -   0.65926
train -   0.80807   |   valid -   0.75373
train -   0.80478   |   valid -   0.68657
Average accuracy of Base Decision Tree Model on crossval is 0.72551,
Std is 0.03562
CPU times: user 17.8 ms, sys: 2.19 ms, total: 19.9 ms
Wall time: 55.7 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.

In [15]:
%%time

results_dt = []

for depth in range(7, 30):
    model = DecisionTreeClassifier(max_depth=depth, random_state=21)

    cv_res = cross_validate(
        model,
        X_train,
        y_train,
        cv=skf,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1,
    )

    mean_acc = cv_res["test_score"].mean()
    std_acc = cv_res["test_score"].std()

    results_dt.append(
        {
            "depth": depth,
            "Mean accuracy": round(mean_acc, 5),
            "Std": round(std_acc, 5),
        }
    )
df_res_dt = pd.DataFrame(results_dt)
# print(df_res_dt)

CPU times: user 245 ms, sys: 86.3 ms, total: 332 ms
Wall time: 589 ms


In [16]:
df_sliced = df_res_dt[(df_res_dt["depth"] >= 18) & (df_res_dt["depth"] <= 27)]
df_sliced.style.highlight_max(
    subset=["Mean accuracy"], color="lightgreen", axis=0
).format({"Mean accuracy": "{:.5f}", "Std": "{:.5f}"}).hide_index()

depth,Mean accuracy,Std
18,0.88055,0.02008
19,0.88054,0.02173
20,0.88649,0.02075
21,0.88278,0.01757
22,0.87981,0.01754
23,0.88353,0.01964
24,0.88722,0.02346
25,0.88649,0.02371
26,0.88501,0.02443
27,0.88575,0.02374


2. As a bonus, play with other regularization parameters trying to find the best combination.

In [17]:
%%time

criteria = ["gini", "entropy"]

dt_combo_results = []

for criterion in tqdm(criteria, file=sys.stderr):
    for depth in tqdm(range(18, 27), leave=False, file=sys.stderr):
        for split in tqdm(range(2, 11), leave=False, file=sys.stderr):
            model = DecisionTreeClassifier(
                criterion=criterion,
                max_depth=depth,
                min_samples_split=split,
                random_state=21,
            )

            cv_res = cross_validate(
                model, X_train, y_train, cv=skf, scoring="accuracy", n_jobs=-1
            )

            dt_combo_results.append(
                {
                    "Criterion": criterion,
                    "Max Depth": depth,
                    "Min Samples Split": split,
                    "Mean accuracy": round(cv_res["test_score"].mean(), 5),
                    "Std": round(cv_res["test_score"].std(), 5),
                }
            )

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

... (12 outputs left out)

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

CPU times: user 1.89 s, sys: 742 ms, total: 2.63 s
Wall time: 4.41 s


In [18]:
df_combo = pd.DataFrame(dt_combo_results)
best_model_indices = df_combo.groupby("Criterion")["Mean accuracy"].idxmax()
df_top_models = df_combo.loc[best_model_indices]
print(df_top_models.to_string(index=False))

Criterion  Max Depth  Min Samples Split  Mean accuracy      Std
  entropy         25                  3        0.88651  0.02092
     gini         24                  2        0.88722  0.02346


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [19]:
model_rf = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)

In [20]:
%%time

cv_res_rf = cross_validate(
    model_rf,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)

mean_acc_rf = cv_res_rf["test_score"].mean()
std_acc_rf = cv_res_rf["test_score"].std()

for train_score, valid_score in zip(cv_res_rf["train_score"], cv_res_rf["test_score"]):
    print(f"{'train - ':<10}{train_score:.5f}   |   {'valid - ':<10}{valid_score:.5f}")
print(f"""\033[32mAverage accuracy of Base Random Forest Model on crossval is {mean_acc_rf:.5f},
Std is {std_acc_rf:.5f}\033[0m""")

train -   0.96455   |   valid -   0.88148
train -   0.96208   |   valid -   0.91852
train -   0.96785   |   valid -   0.86667
train -   0.96455   |   valid -   0.89630
train -   0.96538   |   valid -   0.91111
train -   0.96538   |   valid -   0.88148
train -   0.97115   |   valid -   0.91852
train -   0.96867   |   valid -   0.85185
train -   0.97364   |   valid -   0.88060
train -   0.97941   |   valid -   0.86567
Average accuracy of Base Random Forest Model on crossval is 0.88722,
Std is 0.02204
CPU times: user 10.1 ms, sys: 9.58 ms, total: 19.6 ms
Wall time: 118 ms


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.

In [21]:
%%time

rf_combo_results = []

for depth in tqdm(range(10, 26, 2), file=sys.stderr):
    for est in tqdm(range(50, 250, 50), leave=False, file=sys.stderr):
        model = RandomForestClassifier(
            n_estimators=est, max_depth=depth, random_state=21, n_jobs=-1
        )
        cv_res = cross_validate(
            model, X_train, y_train, cv=skf, scoring="accuracy", n_jobs=-1
        )

        rf_combo_results.append(
            {
                "n_estimators": est,
                "Max Depth": depth,
                "Mean accuracy": round(cv_res["test_score"].mean(), 5),
                "Std": round(cv_res["test_score"].std(), 5),
            }
        )

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

CPU times: user 614 ms, sys: 161 ms, total: 775 ms
Wall time: 18.7 s


In [22]:
df_rf = pd.DataFrame(rf_combo_results)
df_sorted = df_rf.sort_values(by="Mean accuracy", ascending=False)

df_sorted.head(5).style.highlight_max(
    subset=["Mean accuracy"], color="lightgreen", axis=0
).format({"Mean accuracy": "{:.5f}", "Std": "{:.5f}"}).hide_index()

n_estimators,Max Depth,Mean accuracy,Std
200,22,0.91395,0.01819
200,20,0.91320,0.01793
100,22,0.91246,0.02121
100,24,0.91171,0.01950
150,24,0.91098,0.02073


*Максимальная точность **оптимизированной модели Random Forest** достигается при n_estimators = 200 и Max Depth = 22. На бО`льшей глубине (=24 при том же n_estimators) accuracy снизилось , а стандартное отклонение выросло*

2. As a bonus, play with other regularization parameters trying to find the best combination.

In [23]:
%%time

rf_final_combo_results = []

for criterion in tqdm(["gini", "entropy"], file=sys.stderr):
    for est in tqdm([150, 200], leave=False, file=sys.stderr):
        for depth in tqdm([18, 20, 22], leave=False, file=sys.stderr):
            for split in tqdm([2, 5], leave=False, file=sys.stderr):
                model = RandomForestClassifier(
                    n_estimators=est,
                    max_depth=depth,
                    criterion=criterion,
                    min_samples_split=split,
                    random_state=21,
                    n_jobs=-1,
                )

                cv_res = cross_validate(
                    model, X_train, y_train, cv=skf, scoring="accuracy", n_jobs=-1
                )

                rf_final_combo_results.append(
                    {
                        "Criterion": criterion,
                        "n_estimators": est,
                        "Max Depth": depth,
                        "Min Samples Split": split,
                        "Mean accuracy": round(cv_res["test_score"].mean(), 5),
                        "Std": round(cv_res["test_score"].std(), 5),
                    }
                )

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

... (10 outputs left out)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

CPU times: user 570 ms, sys: 199 ms, total: 768 ms
Wall time: 20.5 s


In [24]:
df_rf_final = pd.DataFrame(rf_final_combo_results)
df_rf_sorted = df_rf_final.sort_values(by="Mean accuracy", ascending=False)
df_rf_sorted.head(5).style.highlight_max(
    subset=["Mean accuracy"], color="lightgreen", axis=0
).format({"Mean accuracy": "{:.5f}", "Std": "{:.5f}"}).hide_index()

Criterion,n_estimators,Max Depth,Min Samples Split,Mean accuracy,Std
entropy,150,20,2,0.91468,0.01827
gini,200,22,2,0.91395,0.01819
gini,200,20,2,0.91320,0.01793
entropy,200,22,2,0.91246,0.01987
entropy,200,20,2,0.91246,0.01686


*Максимальная точность  достигнута на **финально оптимизированной модели Random Forest** при `n_estimators = 150` и `Max Depth = 20` со стандартным минимальным сплитом `2` сменой критерия `gini` (по умолчанию) на `entropy`.

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [25]:
%%time

best_model = RandomForestClassifier(
    criterion="entropy",
    n_estimators=150,
    max_depth=20,
    random_state=21,
    n_jobs=-1,
)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)
print(
    f"\033[1;32m Accuracy финальной модели Random Forest на тестовой выборке: {final_accuracy:.5f}\033[0m"
)

 Accuracy финальной модели Random Forest на тестовой выборке: 0.93195
CPU times: user 386 ms, sys: 699 ms, total: 1.08 s
Wall time: 663 ms


3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).


In [26]:
print(f"Тип переменной y_pred: {type(y_pred)}")
print(f"Тип переменной y_test:{type(y_test)}")

Тип переменной y_pred: <class 'numpy.ndarray'>
Тип переменной y_test:<class 'pandas.core.series.Series'>


In [27]:
real_stats = y_test.value_counts()
y_pred_series = pd.Series(y_pred, index=y_test.index)
y_pred_errors = y_test[y_test != y_pred_series]
error_stats = y_pred_errors.value_counts()

error_analysis = pd.DataFrame(
    {
        "real_y": real_stats,
        "total_errors": error_stats,
        "errors_percent": (error_stats / real_stats * 100),
    }
)

days_dict = {0: "Пн", 1: "Вт", 2: "Ср", 3: "Чт", 4: "Пт", 5: "Сб", 6: "Вс"}
error_analysis.index = error_analysis.index.map(days_dict)
error_analysis.style.highlight_max(
    subset=["errors_percent"], color="lightgreen", axis=0
).format({"errors_percent": "{:.2f}%", "real_y": "{:0.0f}", "total_errors": "{:0.0f}"})


,real_y,total_errors,errors_percent
Пн,27,7,25.93%
Вт,55,3,5.45%
Ср,30,2,6.67%
Чт,80,2,2.50%
Пт,21,3,14.29%
Сб,54,4,7.41%
Вс,71,2,2.82%


4. Save the model.

In [28]:
joblib.dump(best_model, "../data/best_random_forest_model_ex01.joblib");

In [29]:
notebook_total_time = time.time() - notebook_start_time
minutes = int(notebook_total_time // 60)
seconds = int(notebook_total_time % 60)

display(
    Markdown(
        f"""
---
**Время работы блокнота:** <span style="color:green">**{minutes} мин. {seconds} сек.**</span>

---
"""
    )
)


---
**Время работы блокнота:** <span style="color:green">**1 мин. 9 сек.**</span>

---
